# BGE-M3 Embedding — Part N of 4

**IMPORTANTE:** Cambiar PART_NUMBER abajo (1, 2, 3 o 4) segun la instancia.

- Part 1: chunks_part1.csv.gz (21K chunks)
- Part 2: chunks_part2.csv.gz (21K chunks)
- Part 3: chunks_part3.csv.gz (21K chunks)
- Part 4: chunks_part4.csv.gz (21K chunks)

**Runtime:** GPU > T4

In [ ]:
# === CAMBIAR ESTO SEGUN LA INSTANCIA ===
PART_NUMBER = 1  # Cambiar a 1, 2, 3 o 4
# ========================================

BATCH_SIZE = 16  # Conservador para evitar OOM
SAVE_EVERY = 2000  # Guardar cada 2000 chunks
FILENAME = f'chunks_part{PART_NUMBER}.csv.gz'

In [ ]:
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
# Upload the correct part file
from google.colab import files
import pandas as pd

uploaded = files.upload()  # Upload chunks_partN.csv.gz
df = pd.read_csv(FILENAME, compression='gzip')
print(f'Part {PART_NUMBER}: {len(df)} chunks loaded')
print(f'Sections: {df["section"].value_counts().to_dict()}')

In [ ]:
# Load BGE-M3
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model = SentenceTransformer('BAAI/bge-m3', device=device)
print(f'Model loaded: {model.get_sentence_embedding_dimension()} dims')

In [ ]:
# Generate embeddings with incremental saving
import numpy as np
import time
import os

texts = df['content'].tolist()
chunk_ids = df['chunk_id'].values
all_embeddings = []
start = time.time()
last_save = 0

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    embs = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.append(embs)
    done = i + len(batch)

    # Save incrementally
    if done - last_save >= SAVE_EVERY or done == len(texts):
        partial = np.vstack(all_embeddings)
        np.save(f'embeddings_part{PART_NUMBER}.npy', partial)
        np.save(f'chunk_ids_part{PART_NUMBER}.npy', chunk_ids[:done])
        last_save = done
        elapsed = time.time() - start
        rate = done / elapsed if elapsed > 0 else 0
        eta = (len(texts) - done) / rate if rate > 0 else 0
        print(f'  SAVED {done}/{len(texts)} ({100*done/len(texts):.1f}%) — {rate:.0f} chunks/s — ETA {eta/60:.1f} min')

elapsed = time.time() - start
print(f'\nDone! Part {PART_NUMBER}: {len(np.vstack(all_embeddings))} embeddings in {elapsed/60:.1f} min')

In [ ]:
# Download results
files.download(f'embeddings_part{PART_NUMBER}.npy')
files.download(f'chunk_ids_part{PART_NUMBER}.npy')